In [1]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib
import pandas as pd
from IPython.display import display

# 1. Buat dataset 1 juta data 5D
np.random.seed(42)
n_data = 1_000_000
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)
query = np.random.random((1, dim)).astype(np.float32)
k = 10
faiss.omp_set_num_threads(4)
hasil_perbandingan = []

In [2]:
def bandingkan_model(metric='euclidean'):
    # Data dan query sama untuk ketiga model.
    data = X.copy()
    q = query.copy()
    if metric == 'cosine':
        faiss.normalize_L2(data)
        faiss.normalize_L2(q)

    hasil = []

    # 2. Annoy
    print('=== Annoy ===')
    ann_index = AnnoyIndex(dim, 'euclidean' if metric == 'euclidean' else 'angular')
    ann_index.set_seed(42)
    start = time.perf_counter()
    for i in range(n_data):
        ann_index.add_item(i, data[i])
    ann_index.build(10, n_jobs=4)
    build_time = time.perf_counter() - start

    start = time.perf_counter()
    neighbors = ann_index.get_nns_by_vector(q[0], k, include_distances=True)
    query_time = time.perf_counter() - start
    print('Build time:', build_time, 'detik')
    print('Query time:', query_time, 'detik')
    print('Neighbors:', neighbors[0][:5], '...')
    hasil.append({'Distance Metrics': metric.capitalize(), 'Model': 'Annoy',
                  'Jumlah data': n_data, 'Dimensi': dim, 'Index terdekat': neighbors[0],
                  'Build time (detik)': build_time, 'Query time (detik)': query_time})
    del ann_index

    # 3. FAISS (Flat Index)
    print('\n=== FAISS (Flat Index) ===')
    faiss_index = faiss.IndexFlatL2(dim) if metric == 'euclidean' else faiss.IndexFlatIP(dim)
    start = time.perf_counter()
    faiss_index.add(data)
    build_time = time.perf_counter() - start

    start = time.perf_counter()
    distances, indices = faiss_index.search(q, k)
    query_time = time.perf_counter() - start
    print('Build time:', build_time, 'detik')
    print('Query time:', query_time, 'detik')
    print('Neighbors:', indices[0][:5], '...')
    hasil.append({'Distance Metrics': metric.capitalize(), 'Model': 'FAISS Flat',
                  'Jumlah data': n_data, 'Dimensi': dim, 'Index terdekat': indices[0].tolist(),
                  'Build time (detik)': build_time, 'Query time (detik)': query_time})
    del faiss_index

    # 4. HNSW (hnswlib)
    print('\n=== HNSW (hnswlib) ===')
    hnsw_index = hnswlib.Index(space='l2' if metric == 'euclidean' else 'cosine', dim=dim)
    start = time.perf_counter()
    hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16, random_seed=42)
    hnsw_index.set_num_threads(4)
    hnsw_index.add_items(data)
    build_time = time.perf_counter() - start
    hnsw_index.set_ef(50)

    start = time.perf_counter()
    labels, distances = hnsw_index.knn_query(q, k=k)
    query_time = time.perf_counter() - start
    print('Build time:', build_time, 'detik')
    print('Query time:', query_time, 'detik')
    print('Neighbors:', labels[0][:5], '...')
    hasil.append({'Distance Metrics': metric.capitalize(), 'Model': 'HNSW',
                  'Jumlah data': n_data, 'Dimensi': dim, 'Index terdekat': labels[0].tolist(),
                  'Build time (detik)': build_time, 'Query time (detik)': query_time})
    return hasil

In [3]:
hasil_perbandingan.extend(bandingkan_model('euclidean'))

=== Annoy ===
Build time: 9.347428200002469 detik
Query time: 0.0001542999998491723 detik
Neighbors: [901095, 495168, 561017, 163986, 185734] ...

=== FAISS (Flat Index) ===
Build time: 0.00872940000044764 detik
Query time: 0.006697900000290247 detik
Neighbors: [901095 495168 561017 163986 185734] ...

=== HNSW (hnswlib) ===
Build time: 67.99180070000148 detik
Query time: 0.0001467000001866836 detik
Neighbors: [901095 495168 561017 163986 185734] ...


In [4]:
hasil_perbandingan.extend(bandingkan_model('cosine'))

=== Annoy ===
Build time: 13.232815899998968 detik
Query time: 0.00013420000323094428 detik
Neighbors: [653774, 135588, 561017, 901095, 211473] ...

=== FAISS (Flat Index) ===
Build time: 0.014762300001166295 detik
Query time: 0.00243649999902118 detik
Neighbors: [653774 135588 561017 901095 211473] ...

=== HNSW (hnswlib) ===
Build time: 60.13265660000252 detik
Query time: 8.600000001024455e-05 detik
Neighbors: [653774 135588 561017 901095 211473] ...


In [5]:
tabel_perbandingan = pd.DataFrame(hasil_perbandingan)
with pd.option_context('display.max_colwidth', None, 'display.float_format', '{:.8f}'.format):
    display(tabel_perbandingan)

,Distance Metrics,Model,Jumlah data,Dimensi,Index terdekat,Build time (detik),Query time (detik)
0,Euclidean,Annoy,1000000,5,"[901095, 495168, 561017, 163986, 185734, 708238, 134961, 859369, 645966, 197596]",9.34742820,0.00015430
1,Euclidean,FAISS Flat,1000000,5,"[901095, 495168, 561017, 163986, 185734, 708238, 134961, 859369, 645966, 197596]",0.00872940,0.00669790
2,Euclidean,HNSW,1000000,5,"[901095, 495168, 561017, 163986, 185734, 708238, 134961, 859369, 645966, 197596]",67.99180070,0.00014670
3,Cosine,Annoy,1000000,5,"[653774, 135588, 561017, 901095, 211473, 163986, 817300, 314338, 495168, 813205]",13.23281590,0.00013420
4,Cosine,FAISS Flat,1000000,5,"[653774, 135588, 561017, 901095, 211473, 163986, 817300, 314338, 495168, 813205]",0.01476230,0.00243650
5,Cosine,HNSW,1000000,5,"[653774, 135588, 561017, 901095, 211473, 163986, 817300, 314338, 495168, 813205]",60.13265660,0.00008600


Untuk cosine, Annoy memakai angular, FAISS memakai inner product pada vektor ternormalisasi, dan HNSW memakai cosine. Urutan tetangganya bisa dibandingkan, tetapi nilai jaraknya berbeda. FAISS Flat mencari secara eksak; Annoy dan HNSW memakai aproksimasi. Waktu build dan pencarian diukur terpisah, tanpa pembuatan data dan normalisasi.